In [1]:
import os
import numpy as np
import scanpy as sc
import pandas as pd

def subsample_random(ref_path, out_dir, min_cells_per_ct=10, total_cells=500, 
                      write=True, ct_key="cell_ontology_class", random_seed=None, repeats=1):
    # Read the input AnnData file
    ref_adata = sc.read_h5ad(ref_path)
    
    # Set random seed for reproducibility if provided
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Extract a unique identifier from the reference file name
    tsp = os.path.basename(ref_path).split('_')[0]
    
    # Create output directory if it doesn't exist
    os.makedirs(out_dir, exist_ok=True)
    
    # Precompute cell type indices for efficiency
    cell_type_indices = ref_adata.obs.groupby(ct_key, observed=False)[[]].apply(lambda x: x.index.to_list()).to_dict()
    
    # Filter out cell types with fewer than min_cells_per_ct occurrences
    valid_cell_types = [ct for ct in cell_type_indices if len(cell_type_indices[ct]) >= min_cells_per_ct]
    all_selected_cts = set()
    proportion_records = []  # Store proportions for all subsampled datasets
    all_adatas = []
    all_paths = []
    
    for i in range(repeats):  # Perform multiple independent subsampling runs
        # Randomly select between 2 and 8 cell types
        num_selected_cts = np.random.randint(2, min(9, len(valid_cell_types) + 1))
        selected_cts = np.random.choice(valid_cell_types, num_selected_cts, replace=False)
        all_selected_cts.update(selected_cts)  # Track all selected cell types
        
        # Determine the maximum allowable cells for each selected cell type
        max_cells_per_ct = {ct: len(cell_type_indices[ct]) for ct in selected_cts}
        
        # Generate random proportions and adjust to fit available cells
        while True:
            proportions = np.random.dirichlet(np.ones(num_selected_cts))
            proportions = np.round(proportions, 2)
            proportions /= proportions.sum()
            required_cells = {ct: min(max(min_cells_per_ct, int(total_cells * prop)), max_cells_per_ct[ct])
                              for ct, prop in zip(selected_cts, proportions)}
            if sum(required_cells.values()) == total_cells:
                break
        
        # Generate output file path including the random seed in the filename
        path = os.path.join(out_dir, f"{tsp}_uni_rep{i+1}_seed{random_seed}.h5ad")
        
        # Sample cells based on adjusted proportions
        sampled_indices = []
        for cell_type, num_samples in required_cells.items():
            sampled_indices.extend(np.random.choice(cell_type_indices[cell_type], size=num_samples, replace=False))
        
        # Subset the data to the sampled indices and create a new AnnData object
        adata_sub = ref_adata[sampled_indices].copy()
        
        # Store the proportions used for this subsampling
        proportion_records.append({"Donor": tsp, "Repeat": i + 1, "RandomSeed": random_seed, "TotalCells": total_cells, **required_cells})
        
        # Optionally write the subsampled data to disk
        if write:
            adata_sub.write(path)
            print("Write AnnData to", path)
        
        # Store results from this repeat
        all_adatas.append(adata_sub)
        all_paths.append(path)
    
    # Convert proportion records to DataFrame
    proportion_df = pd.DataFrame(proportion_records)
    
    # Ensure all columns for selected cell types exist, filling missing values with 0
    proportion_df = proportion_df.reindex(columns=["Donor", "Repeat", "RandomSeed", "TotalCells"] + sorted(all_selected_cts), fill_value=0)
    
    # Save proportions to a tab-separated text file
    proportion_file = os.path.join(out_dir, f"{tsp}_subsampling_proportions.txt")
    proportion_df.to_csv(proportion_file, sep='\t', index=False)
    print("Saved proportions to", proportion_file)
    
    return all_adatas, all_paths


In [2]:
# Define input parameters
ref_path = "05_Merged-PerDonor-QC-h5ad/TSP25_merged_processed.h5ad"  # Path to the input AnnData file
out_dir = "output-random"  # Directory to save the subsampled files
min_cells_per_ct = 50  # Minimum number of cells per cell type
total_cells = 2000  # Total number of cells to sample
write = True  # Whether to save the subsampled files
ct_key = "cell_ontology_class"  # Key in `obs` containing cell type annotations
random_seed = 20  # Set seed for reproducibility
repeats = 2  # Number of independent subsampling runs

# Run the subsampling function
subsampled_adatas, subsampled_paths = subsample_random(
    ref_path, out_dir, min_cells_per_ct=min_cells_per_ct, total_cells=total_cells,
    write=write, ct_key=ct_key, random_seed=random_seed, repeats=repeats
)

# The function returns:
# - `subsampled_adatas`: A list of AnnData objects containing the subsampled datasets
# - `subsampled_paths`: A list of file paths where the datasets were saved

print("Subsampled datasets saved at:", subsampled_paths)

Write AnnData to output-random/TSP25_uni_rep1_seed20.h5ad
Write AnnData to output-random/TSP25_uni_rep2_seed20.h5ad
Saved proportions to output-random/TSP25_subsampling_proportions.txt
Subsampled datasets saved at: ['output-random/TSP25_uni_rep1_seed20.h5ad', 'output-random/TSP25_uni_rep2_seed20.h5ad']


In [4]:
adata = sc.read_h5ad('output-random/TSP25_uni_rep2_seed20.h5ad')
adata.obs["cell_ontology_class"].value_counts()

cell_ontology_class
thymic fibroblast type 2           1900
alveolar type 2 fibroblast cell     100
Name: count, dtype: int64